In [3]:
!pip install mlflow boto3 awscli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/

In [ ]:
!aws configure

In [5]:
import mlflow
# setup up the mlflow tracking server
mlflow.set_tracking_uri("http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/")

In [ ]:
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

In [7]:
from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [11]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36627, 2)

In [13]:
import pickle

# step 1: function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 3)
    max_features = 10000

    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']  # ✅ FIX: added stratify
    )

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    if imbalance_method == 'class_weights':
        class_weight = 'balanced'  # ✅ FIX: 'balance' → 'balanced'
    else:
        class_weight = None
        if imbalance_method == 'oversampling':  # ✅ FIX: typo 'oversmapling' → 'oversampling'
            smote = SMOTE(random_state=42)
            X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'adasyn':
            adasyn = ADASYN(random_state=42)
            X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'smote_enn':
            smote_enn = SMOTEENN(random_state=42)
            X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    with mlflow.start_run() as run:
        # ✅ FIX: 'mlflow.runname' → 'mlflow.runName', typo 'RamdomForest' → 'RandomForest', 'TIFDF' → 'TFIDF'
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigram")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")
        # ✅ FIX: missing comma after closing quote in set_tag call
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # ✅ FIX: 'n_estimator' → 'n_estimators' (consistent naming)
        n_estimators = 200
        max_depth = 15
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        # ✅ FIX: 'mlflow.lot_param' → 'mlflow.log_param', 'imbalace_method' → 'imbalance_method'
        mlflow.log_param("imbalance_method", imbalance_method)

        # ✅ FIX: 'n_estimator' → 'n_estimators', added random_state for reproducibility
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                       class_weight=class_weight, random_state=42)
        model.fit(X_train_vec, y_train)

        y_pred = model.predict(X_test_vec)
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # ✅ FIX: 'isinstance(label, dict)' → 'isinstance(metrics, dict)' (was checking wrong variable)
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"classification_report_{label}_{metric}", value)

        # log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        # ✅ FIX: typo 'Triagrams' → 'Trigrams'
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig(confusion_matrix_filename)
        mlflow.log_artifact(confusion_matrix_filename)
        plt.close()  # ✅ FIX: removed duplicate plt.savefig/log_artifact and misplaced plt.close()

        # ✅ FIX: 'mlflow.slearn' → pickle-based saving to avoid server version mismatch
        model_filename = f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}.pkl"
        with open(model_filename, "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact(model_filename)


mlflow.set_experiment("RF Imbalance Handling")  # ✅ FIX: added missing set_experiment

# ✅ FIX: 'oversmapling' → 'oversampling' to match the if-condition inside the function
imbalance_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']
for method in imbalance_methods:
    run_imbalanced_experiment(method)

2026/04/12 14:58:44 INFO mlflow.tracking.fluent: Experiment with name 'RF Imbalance Handling' does not exist. Creating a new experiment.


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigram at: http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/#/experiments/552328580527200762/runs/53173bd2b055413793432f1cd4401a00
🧪 View experiment at: http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/#/experiments/552328580527200762
🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigram at: http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/#/experiments/552328580527200762/runs/e8719ef86833483ea92e425a4a4e990a
🧪 View experiment at: http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/#/experiments/552328580527200762
🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigram at: http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/#/experiments/552328580527200762/runs/d3d73fdc0cb24569ab3e5262061eba37
🧪 View experiment at: http://ec2-98-93-179-250.compute-1.amazonaws.com:5000/#/experiments/552328580527200762
🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigram at: http://ec2-98-93-179-250.compute-1.amazona